In [1]:
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets

from PIL import Image

import torch.nn.functional as F
from torch import nn

import math
import torch.nn as nn

import hashlib
import urllib
import warnings
from typing import Any
from pkg_resources import packaging

from torchvision.transforms import Compose, Resize, ToTensor, Normalize

import torch.utils.data
import torch.utils.data.distributed

/tmp/ipykernel_1697/2767196239.py:17: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import packaging


In [2]:
%cd /content
!wget -O imagenet-r.tar https://people.eecs.berkeley.edu/~hendrycks/imagenet-r.tar
!tar -xf imagenet-r.tar

/content
--2026-06-01 08:30:43--  https://people.eecs.berkeley.edu/~hendrycks/imagenet-r.tar
Resolving people.eecs.berkeley.edu (people.eecs.berkeley.edu)... 128.32.244.190
Connecting to people.eecs.berkeley.edu (people.eecs.berkeley.edu)|128.32.244.190|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2191079936 (2.0G) [application/x-tar]
Saving to: ‘imagenet-r.tar’

imagenet-r.tar      100%[===================>]   2.04G  20.4MB/s    in 1m 43s  

2026-06-01 08:32:26 (20.4 MB/s) - ‘imagenet-r.tar’ saved [2191079936/2191079936]



In [3]:
import torchvision.datasets as dset

dataset = dset.ImageFolder(root="/content/imagenet-r")
print(f"클래스 수: {len(dataset.classes)}")
print(f"전체 이미지 수: {len(dataset)}")

클래스 수: 200
전체 이미지 수: 30000


In [4]:
!pip install ftfy regex tqdm git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-w1sq7btc
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-w1sq7btc
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=3a89931f34014b7b2685d3b1f9cb8ea74d555bd4489c9204db1101354889004a
  Stored in directory: /tmp/pip-ephem-wheel-cache-9ct4d34c/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [5]:
def make_views(image, n_views):
    normalize = transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    )
    base_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        normalize,
    ])
    crop_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.ToTensor(),
        normalize,
    ])

    original = base_transform(image).unsqueeze(0)
    crops = torch.stack([crop_transform(image) for _ in range(n_views)])
    all_views = torch.cat([original, crops], dim=0)
    return all_views

In [6]:
class MakeViewsTransform:
    def __init__(self, n_views, resolution):
        self.n_views = n_views
        # Note: self.resolution is ignored by the make_views function as it hardcodes 224.
        self.resolution = resolution
        # ensure PIL image is RGB before passing to make_views
        self.to_rgb = transforms.Lambda(lambda img: img.convert("RGB"))

    def __call__(self, image):
        # Ensure image is RGB PIL before passing to make_views
        pil_image = self.to_rgb(image)
        # make_views is now part of this file
        return make_views(pil_image, self.n_views)

In [7]:
# 가우시안 커널
def gaussian_kernel(mu, bandwidth, datapoints):
    dist = torch.norm(datapoints - mu,dim=-1, p=2)
    density = torch.exp(-dist**2/(2*bandwidth**2))
    return density

In [8]:
# MTA (inlierness score 계산 + mode m 찾기)

def solve_mta(model, inputs, args):

    with torch.no_grad():
        with torch.cuda.amp.autocast():
            image_features = model.encode_image(inputs)

    image_features = image_features.float()
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    lambda_y = args.lambda_y
    lambda_q = args.lambda_q
    max_iter = 5
    temperature = 1

    batch_size = image_features.shape[0]

    # bandwidth
    dist = torch.cdist(image_features, image_features)
    sorted_dist, _ = torch.sort(dist, dim=1)

    k = int(0.3 * (image_features.shape[0]-1))
    k = max(1, k)

    selected_distances = sorted_dist[:, 1:k+1]**2  # exclude the distance to the point itself
    mean_distance = torch.mean(selected_distances, dim=1)
    bandwidth = torch.sqrt(0.5 * mean_distance)

    affinity_matrix = (image_features @ image_features.t() / temperature).softmax(1)

    # Inlierness scores initialization: uniform
    y = torch.ones(batch_size, device=image_features.device)/batch_size

    # Mode initialization: original image embedding
    mode_init = image_features[0]
    mode = mode_init

    convergence = False
    th = 1e-6
    iter = 0

    while not convergence:

        # Inlierness step
        density = gaussian_kernel(mode, bandwidth, image_features)

        convergence_inlierness = False
        i = 0
        while not convergence_inlierness:
            i+=1
            old_y = y
            weighted_affinity = affinity_matrix * y.unsqueeze(0)
            y = F.softmax(1/lambda_y * (density + lambda_q * torch.sum(weighted_affinity, dim=1)), dim=-1)

            if torch.norm(old_y - y)<th or i>= max_iter:
                convergence_inlierness = True

        # Mode step #
        convergence_mode = False
        i=0
        while not convergence_mode:
            i+=1
            old_mode = mode
            density = gaussian_kernel(mode, bandwidth, image_features)
            weighted_density = density *  y
            mode = torch.sum(weighted_density.unsqueeze(1)* image_features, dim=0)/torch.sum(weighted_density)
            mode /= mode.norm(p=2, dim=-1)

            if torch.norm(old_mode - mode)<th or i>= max_iter:
                convergence_mode = True

        iter +=1
        if iter >= max_iter:
            convergence = True

# 테스트 용으로 return 값이 3개. mode만 쓰려면 return mode로 수정
    return mode, image_features, y


In [10]:
from types import SimpleNamespace
import clip

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

args = SimpleNamespace(
    lambda_y=0.2,
    lambda_q=4
)

dataset = dset.ImageFolder(root="/content/imagenet-r")

image_path, label = dataset.samples[0]
image = Image.open(image_path).convert("RGB")

inputs = make_views(image, 127)
inputs = inputs.to(device)

mode, image_features, y = solve_mta(model, inputs, args)

print(f"생성된 이미지 view 개수와 크기: {inputs.shape}")
print(f"CLIP 이미지 feature 형태: {image_features.shape}")
print(f"MTA가 계산한 대표 임베딩(mode)의 형태: {mode.shape}")
print(f"대표 임베딩의 벡터 크기(norm): {mode.norm().item():.4f}")

print(f"정답 라벨 번호: {label}")

# 1. 원본 view와 mode의 cosine similarity
original_sim = torch.sum(mode * image_features[0]).item()
print(f"원본 view와 mode의 코사인 유사도: {original_sim:.4f}")

# 2. 모든 augmented view와 mode의 similarity
sims = image_features @ mode

print(f"전체 view와 mode의 평균 유사도: {sims.mean().item():.4f}")
print(f"전체 view와 mode의 최대 유사도: {sims.max().item():.4f}")
print(f"전체 view와 mode의 최소 유사도: {sims.min().item():.4f}")
print(f"전체 view와 mode의 유사도 표준편차: {sims.std().item():.4f}")

# 3. mode와 단순 평균 feature 비교
mean_feature = image_features.mean(dim=0)
mean_feature = mean_feature / mean_feature.norm(dim=-1, keepdim=True)

mode_mean_sim = torch.sum(mode * mean_feature).item()
print(f"mode와 단순 평균 feature의 cosine similarity: {mode_mean_sim:.4f}")

# 4. inlierness score 확인
print(f"inlierness score y의 형태: {y.shape}")
print(f"inlierness score의 전체 합: {y.sum().item():.4f}")
print(f"가장 큰 inlierness score: {y.max().item():.4f}")
print(f"가장 작은 inlierness score: {y.min().item():.4f}")

topk = torch.topk(y, k=5)

print(f"상위 5개 inlier view index: {topk.indices.tolist()}")
print(f"상위 5개 inlier score: {[round(v, 4) for v in topk.values.tolist()]}")

/tmp/ipykernel_1697/592006928.py:6: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


생성된 이미지 view 개수와 크기: torch.Size([128, 3, 224, 224])
CLIP 이미지 feature 형태: torch.Size([128, 512])
MTA가 계산한 대표 임베딩(mode)의 형태: torch.Size([512])
대표 임베딩의 벡터 크기(norm): 1.0000
정답 라벨 번호: 0
원본 view와 mode의 코사인 유사도: 0.9684
전체 view와 mode의 평균 유사도: 0.9244
전체 view와 mode의 최대 유사도: 0.9887
전체 view와 mode의 최소 유사도: 0.8011
전체 view와 mode의 유사도 표준편차: 0.0439
mode와 단순 평균 feature의 cosine similarity: 0.9839
inlierness score y의 형태: torch.Size([128])
inlierness score의 전체 합: 1.0000
가장 큰 inlierness score: 0.0407
가장 작은 inlierness score: 0.0017
상위 5개 inlier view index: [56, 89, 13, 82, 93]
상위 5개 inlier score: [0.0407, 0.0389, 0.0331, 0.0316, 0.0304]
